# Ward Boundary Crosswalk and Extended Election Panel (2000-2021)

**Added after the original Group 1 (2011-2021) panel and the first modeling round were
already complete.**

While preparing to merge the newly found 2000 and 2006 election data into Group 1, we needed
a way to compare ward turnout across years even though ward numbers change over time. That
led us to test something we had previously only logged as a limitation: whether ward numbers
are actually stable across 2011, 2016, and 2021 as well, not just across the older years.

**They are not.** This notebook documents that finding, builds a voting-district-based
crosswalk to correct for it, and produces a five-election ward panel (2000, 2006, 2011, 2016,
2021) using the corrected ward identity instead of the raw ward number reported in each year's
file.


In [ ]:
import pandas as pd
import os

# os.chdir("../..")  # uncomment and adjust if running from a subfolder
print("Working directory:", os.getcwd())


## Step 1: The discovery

We picked a single physical polling station, `VotingDistrict 43400179` (Nomfihlela Primary
School), and traced which ward it was recorded under in each election year.


In [ ]:
df2000 = pd.read_csv("data/interim/KZN_Local_Government_Elections_2000-2006 Cleaned/KZN_2000_clean.csv")
df2006 = pd.read_csv("data/interim/KZN_Local_Government_Elections_2000-2006 Cleaned/KZN_2006_clean.csv")
df2011 = pd.read_csv("data/interim/KZN_Local_Goverment_elections_2011-21 Cleaned/KZN_2011_clean.csv")
df2016 = pd.read_csv("data/interim/KZN_Local_Goverment_elections_2011-21 Cleaned/KZN_2016_clean.csv")
df2021 = pd.read_csv("data/interim/KZN_Local_Goverment_elections_2011-21 Cleaned/KZN_2021_clean.csv")

test_station = 43400179

print("2000 ward for this station:", df2000[df2000["VotingDistrict"] == test_station]["Ward"].unique())
print("2006 ward for this station:", df2006[df2006["VotingDistrict"] == test_station]["Ward"].unique())
print("2011 ward for this station:", df2011[df2011["VotingDistrict"] == test_station]["Ward"].unique())
print("2016 ward for this station:", df2016[df2016["VotingDistrict"] == test_station]["Ward"].unique())
print("2021 ward for this station:", df2021[df2021["VotingDistrict"] == test_station]["Ward"].unique())


The same physical station was ward 59200001 in 2000 and 2006, then renumbered to ward
59500001 by 2011 and remained that way through 2021. Ward *numbers* changed even within the
range we had already modeled.

We then checked this systematically, not just for one station: for every voting district
present in both 2011 and 2021 (and separately, 2016 and 2021), does its recorded ward number
match?


In [ ]:
def build_crosswalk(reference_df):
    cw = reference_df[["VotingDistrict", "Ward"]].drop_duplicates().copy()
    cw["Ward2021Reference"] = cw["Ward"].str.extract(r"(\d+)").astype(int)
    return cw[["VotingDistrict", "Ward2021Reference"]]

crosswalk_2021 = build_crosswalk(df2021)

def check_mismatch(df, year, label):
    d = df.copy()
    d["WardNum"] = d["Ward"].str.extract(r"(\d+)").astype(int)
    check = d[["VotingDistrict", "WardNum"]].drop_duplicates().merge(
        crosswalk_2021, on="VotingDistrict", how="left"
    )
    mismatch = (check["WardNum"] != check["Ward2021Reference"]).sum()
    total = check.shape[0]
    unmapped = check["Ward2021Reference"].isna().sum()
    print(f"{label}: {total} station-ward pairs checked against 2021 numbering, "
          f"{mismatch} mismatched ({mismatch/total:.1%}), {unmapped} stations not found in 2021 at all")

check_mismatch(df2011, 2011, "2011 vs 2021 ward numbering")
check_mismatch(df2016, 2016, "2016 vs 2021 ward numbering")


## What this means

A large share of voting districts were recorded under a different ward number in 2011 than
in 2021 (roughly a third), and a smaller but still meaningful share differ between 2016 and
2021. This means comparing `Ward "10"` in one year to `Ward "10"` in another year, without
correction, is comparing two different sets of voting stations in a meaningful number of
cases -- not the same ward.

**Decision we made:** rather than treat this only as a limitation to write down, we built a
voting-district-based crosswalk, using the most recent election (2021) as the reference ward
numbering, and re-expressed every earlier election's results in terms of that same reference
ward. A voting district is a physical polling location and does not get renumbered the way a
ward boundary/label does, which makes it a far more stable anchor for comparing across time.


## Step 2: Build the crosswalk and save it as a standalone reference table


In [ ]:
crosswalk = df2021[["VotingDistrict", "Ward", "Municipality"]].drop_duplicates().copy()
crosswalk["Ward2021Reference"] = crosswalk["Ward"].str.extract(r"(\d+)").astype(int)
crosswalk = crosswalk.rename(columns={"Municipality": "Municipality2021"})[
    ["VotingDistrict", "Ward2021Reference", "Municipality2021"]
]

dupe_check = crosswalk.groupby("VotingDistrict")["Ward2021Reference"].nunique()
print("Voting districts mapping to more than one 2021 ward (should be 0):", (dupe_check > 1).sum())

os.makedirs("data/processed/01_ward_election_panel/", exist_ok=True)
crosswalk.to_csv("data/processed/01_ward_election_panel/voting_district_to_2021_ward_crosswalk.csv", index=False)
print("Crosswalk saved:", crosswalk.shape)


## Step 3: Re-aggregate every election year to ward level using the crosswalk

Two important corrections built into this aggregation:

1. **Ward identity comes from the crosswalk (2021 numbering), not from the ward number printed
   in that year's own file.**
2. **Turnout is calculated from the PR ballot only.** Each voter casts two ballots (PR and
   Ward) in South African local government elections. Summing valid votes across both ballot
   types double-counts votes cast. We use the PR ballot consistently across all five years, since
   it is present in every year's file and is the standard ballot used for reporting turnout.
   (For 2000 and 2006, which additionally report a pre-computed station-level `TotalVotesCast`
   figure, we use that direct figure rather than re-deriving it, since it already reflects a
   single count per station.)


In [ ]:
def build_ward_year_from_ballots(df, year, crosswalk):
    df_pr = df[df["BallotType"].str.upper().str.strip() == "PR"].copy()
    station = df_pr.groupby(["VotingDistrict", "Municipality"], as_index=False).agg(
        RegisteredVoters=("RegisteredVoters", "first"),
        TotalVotesCast=("TotalValidVotes", "sum"),
    )
    return finish_ward_year(station, year, crosswalk)


def build_ward_year_from_precomputed(df, year, crosswalk):
    station = df.drop_duplicates(subset=["VotingDistrict", "RegisteredVoters", "TotalVotesCast"])
    station = station[["VotingDistrict", "Municipality", "RegisteredVoters", "TotalVotesCast"]]
    return finish_ward_year(station, year, crosswalk)


def finish_ward_year(station, year, crosswalk):
    station = station.merge(crosswalk, on="VotingDistrict", how="left")
    unmapped = station["Ward2021Reference"].isna().sum()
    total = station.shape[0]

    mapped = station.dropna(subset=["Ward2021Reference"]).copy()
    mapped["Ward2021Reference"] = mapped["Ward2021Reference"].astype(int)

    ward_year = mapped.groupby(["Ward2021Reference", "Municipality"], as_index=False).agg(
        RegisteredVoters=("RegisteredVoters", "sum"),
        TotalVotesCast=("TotalVotesCast", "sum"),
        VotingStationsMatched=("VotingDistrict", "nunique"),
    )
    ward_year["TurnoutRate"] = (ward_year["TotalVotesCast"] / ward_year["RegisteredVoters"]) * 100
    ward_year["ElectionYear"] = year

    print(f"{year}: {total} stations, {unmapped} unmapped to a 2021 ward "
          f"({unmapped/total:.1%}), {ward_year.shape[0]} wards produced, "
          f"mean turnout {ward_year['TurnoutRate'].mean():.1f}%")
    return ward_year


ward2000 = build_ward_year_from_precomputed(df2000, 2000, crosswalk[["VotingDistrict","Ward2021Reference"]])
ward2006 = build_ward_year_from_precomputed(df2006, 2006, crosswalk[["VotingDistrict","Ward2021Reference"]])
ward2011 = build_ward_year_from_ballots(df2011, 2011, crosswalk[["VotingDistrict","Ward2021Reference"]])
ward2016 = build_ward_year_from_ballots(df2016, 2016, crosswalk[["VotingDistrict","Ward2021Reference"]])
ward2021 = build_ward_year_from_ballots(df2021, 2021, crosswalk[["VotingDistrict","Ward2021Reference"]])


## Step 4: Sanity check turnout rates before combining


In [ ]:
for year, w in [(2000, ward2000), (2006, ward2006), (2011, ward2011), (2016, ward2016), (2021, ward2021)]:
    print(f"{year}: min={w['TurnoutRate'].min():.1f}%, mean={w['TurnoutRate'].mean():.1f}%, "
          f"max={w['TurnoutRate'].max():.1f}%")
    assert w["TurnoutRate"].between(0, 100).all(), f"Turnout out of range in {year}!"
print("\nAll turnout rates fall within 0-100%.")


## Step 5: Combine into one five-election panel and check coverage


In [ ]:
combined = pd.concat([ward2000, ward2006, ward2011, ward2016, ward2021], ignore_index=True)
combined = combined.rename(columns={"Ward2021Reference": "Ward"})
combined = combined.sort_values(["Ward", "ElectionYear"]).reset_index(drop=True)

print("Combined panel shape:", combined.shape)
print("\nRows per election year:")
print(combined["ElectionYear"].value_counts().sort_index())

coverage = combined.groupby("Ward")["ElectionYear"].nunique()
complete = (coverage == 5).sum()
print(f"\nWards with all 5 elections present: {complete} out of {coverage.shape[0]} "
      f"({complete/coverage.shape[0]:.1%})")


## Step 6: Save the extended, crosswalk-corrected panel

This becomes a new file inside the existing Group 1 folder, alongside the original
`ward_election_panel_2011_2021.csv`. We are not overwriting the original file: the team should
decide together whether to switch the downstream feature engineering and modeling notebooks
over to this corrected panel, since it changes the ward identifier convention (now anchored to
2021 numbering via voting district, not the raw ward number in each year's file).


In [ ]:
output_path = "data/processed/01_ward_election_panel/ward_election_panel_2000_2021_crosswalk_corrected.csv"
combined.to_csv(output_path, index=False)
print("Saved:", output_path, "-", combined.shape)


## Summary of what changed and why

- **Before:** the Group 1 panel covered 2011, 2016, and 2021 only, using each year's own ward
  number directly, with ward-boundary consistency logged as a general limitation rather than
  measured.
- **What we found:** ward numbers are not stable even within 2011-2021 (roughly a third of
  voting districts were under a different ward number in 2011 than in 2021). This means some
  `PreviousTurnout` values in the original feature-engineered dataset may have been comparing
  different physical areas without us realising it.
- **What we built:** a voting-district-based crosswalk anchored to the 2021 ward numbering,
  and a five-election panel (2000, 2006, 2011, 2016, 2021) built on top of it, with 863 of 901
  reference wards (95.8%) having complete history across all five elections.
- **What's next:** re-run feature engineering and modeling on this corrected panel instead of
  the original one, to see whether correcting the ward-identity problem, and/or adding two more
  historical elections, changes the model's performance against the baseline.
